# 09 Graph v2, step B0: address-map check, domain exclusions, edge counts, timing probe, edge-causality check
Pre-registered in `docs/PROJECT_LOG.md` section 6f. Nothing is fitted for evaluation here; the timing probe trains for 10 epochs only to measure seconds per epoch. K = 5 unless the probe says stage 1 cannot fit (then K = 3); K is never changed after fold results.

In [1]:
import os, time, json, resource
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from itgnn.comms_graph import (PC, DOM, EMAIL, adjacency, build_from_inputs, build_graph_v2, edge_table, load_inputs, node_features_v2, shuffle_edges)
from itgnn.data import load_config
from itgnn.evaluation import make_folds
from itgnn.weekly_gnn import train_scores
from itgnn.weekly_graph import fold_masks

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/config.yaml"); SEED = cfg["seed"]; gcfg = cfg["gnn"]
LOG = open(os.environ.get("PROBE_LOG", os.devnull), "a"); T0 = time.time()
def tick(s): print(f"[{s}] {time.time() - T0:.0f}s", file=LOG, flush=True)
pd.options.display.float_format = "{:,.3f}".format; pd.options.display.width = 200
torch.set_num_threads(os.cpu_count())
inp = load_inputs(cfg); d = inp["d"]; labels = d["labels"]
print("threads:", torch.get_num_threads(), "| torch", torch.__version__); tick("inputs loaded")

/home/nothing/insider-threat-gnn/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


threads: 20 | torch 2.15.0+cpu


## 1. Address-to-user map (static identity from LDAP) and the 90% rule

In [2]:
cov = inp["cov"]; print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in cov.items()})
print("addresses in the LDAP table:", len(inp["addr"]), "| ambiguous:", len(inp["amb"]))
print(f"RULE: both shares must be >= 90%  ->  email edges {'KEPT' if inp['use_email'] else 'DROPPED'}")
dom_total = inp["long"]["recip"].str.endswith("@dtaa.com").mean()
print(f"internal share of all recipient entries: {dom_total:.3f}")

{'internal entries': 3547274, 'internal addresses': 1000, 'entries mapped to exactly one user': 1.0, 'addresses mapped to exactly one user': 1.0, 'ambiguous addresses (>1 user)': 0, 'internal addresses not in LDAP': 0}
addresses in the LDAP table: 1000 | ambiguous: 0
RULE: both shares must be >= 90%  ->  email edges KEPT
internal share of all recipient entries: 0.582


## 2. Domains and aggregate rows removed from the edges

In [3]:
print({k: v for k, v in inp["drop_rep"].items()})

{'domains total': 475, 'category domains': 28, 'excluded domains': 5, '< 20 users': 18, 'domains dropped (union)': 38, 'domains kept': 437, 'aggregate rows dropped': 232068, 'aggregate rows kept': 3794409, 'aggregate rows total': 4026477}


## 3. Edge counts and timing probe per candidate K

In [4]:
fold0 = [f for f in make_folds(d["users"], labels, cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"], SEED) if f["repeat"] == 0][0]
EP = {"mlp": cfg["gnn"]["epochs"]["mlp"], "sage": cfg["gnn"]["epochs"]["sage"]}   # step-5 counts, placeholders for the projection only
def probe(g, A, conv, epochs=10):
    fit, _ = fold_masks(g, fold0["train"], fold0["test"])
    x, _ = node_features_v2(g, fit); y = g.rows["label"].to_numpy()
    train_scores(x, A, y, np.flatnonzero(fit), 2, conv, gcfg, SEED)       # warm-up
    t = time.time(); train_scores(x, A, y, np.flatnonzero(fit), epochs, conv, gcfg, SEED); return (time.time() - t) / epochs
chosen = None; info = {}
for K in (5, 3):
    t = time.time(); g = build_from_inputs(inp, K, K); tb = time.time() - t
    cnt = {n: int((g.etype == c).sum()) for n, c in [("user-PC", PC), ("user-domain", DOM), ("user-user email", EMAIL)]}
    A = adjacency(g); ts = time.time(); gs, share = shuffle_edges(g, SEED); As = adjacency(gs); tshuf = time.time() - ts
    t_mlp, t_sage = probe(g, A, "mlp"), probe(g, A, "wsage")
    stage1 = 5 * (EP["mlp"] * t_mlp + 2 * EP["sage"] * t_sage + tshuf) / 60
    pilot = 3 * 300 * (t_mlp + t_sage) * 1.3 / 60
    info[K] = dict(K=K, nodes_user=g.n_user, nodes_pc=g.n_pc, nodes_domain=g.n_dom, undirected_edges=cnt, nnz=int(A._nnz()), build_s=round(tb, 1),
                   email_selfloop_share_after_shuffle=round(share, 4), s_per_epoch_mlp=round(t_mlp, 3), s_per_epoch_wsage=round(t_sage, 3),
                   projected_stage1_min=round(stage1, 1), projected_pilot_min=round(pilot, 1), peak_rss_gb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20, 2))
    print(json.dumps(info[K], indent=1)); tick(f"K={K} probed")
    if stage1 <= 25 and stage1 + pilot <= 60: chosen = K; break
print("CHOSEN K:", chosen, "(None = stop and ask: even K = 3 projects beyond the budget)")

/home/nothing/insider-threat-gnn/src/itgnn/comms_graph.py:183: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:916.)
  return torch.sparse_csr_tensor(torch.from_numpy(A.indptr.astype(np.int64)), torch.from_numpy(A.indices.astype(np.int64)),
/home/nothing/insider-threat-gnn/src/itgnn/comms_graph.py:183: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  return torch.sparse_csr_tensor(torch.from_numpy(A.indptr.

{
 "K": 5,
 "nodes_user": 67238,
 "nodes_pc": 67375,
 "nodes_domain": 24999,
 "undirected_edges": {
  "user-PC": 112408,
  "user-domain": 336159,
  "user-user email": 322392
 },
 "nnz": 1452830,
 "build_s": 3.3,
 "email_selfloop_share_after_shuffle": 0.0011,
 "s_per_epoch_mlp": 0.376,
 "s_per_epoch_wsage": 0.489,
 "projected_stage1_min": 24.2,
 "projected_pilot_min": 16.9,
 "peak_rss_gb": 4.06
}
CHOSEN K: 5 (None = stop and ask: even K = 3 projects beyond the budget)


## 4. Edge-causality check on real data (logs truncated at the end of 15 sampled weeks)

In [5]:
if chosen is not None:
    K = chosen; g = build_from_inputs(inp, K, K); et = edge_table(g)
    dnode0 = g.n_user + g.n_pc
    dom_full = pd.DataFrame(g.dom_feat, columns=["a", "b", "c"]).assign(domain=g.dom_name, week_start=g.node_week[dnode0:])
    weeks = np.sort(inp["full"]["week_start"].unique()); rng = np.random.default_rng(SEED)
    pick = sorted(rng.choice(weeks[8:], 15, replace=False))
    full, logon, agg, long = inp["full"], d["logon"], inp["agg"], inp["long"]; bad = []
    for w in pick:
        w = pd.Timestamp(w); end = w + pd.Timedelta(days=7)
        gt = build_graph_v2(full[full["week_start"] <= w], logon[logon["date"] < end], agg[agg["week_start"] <= w],
                            long[long["date"] < end] if inp["use_email"] else None, inp["addr"], inp["cols"], K, K, frozenset(inp["drop"]))
        a, b = edge_table(gt), et
        a, b = a[a["week_start"] == w].reset_index(drop=True), b[b["week_start"] == w].reset_index(drop=True)
        same_edges = a.equals(b)
        dt = pd.DataFrame(gt.dom_feat, columns=["a", "b", "c"]).assign(domain=gt.dom_name, week_start=gt.node_week[gt.n_user + gt.n_pc:])
        x1 = dt[dt["week_start"] == w].sort_values("domain").reset_index(drop=True); x2 = dom_full[dom_full["week_start"] == w].sort_values("domain").reset_index(drop=True)
        same_dom = x1.equals(x2) or (len(x1) == len(x2) and np.allclose(x1[["a", "b", "c"]], x2[["a", "b", "c"]]) and (x1["domain"] == x2["domain"]).all())
        if not (same_edges and same_dom): bad.append((str(w.date()), len(a), len(b), same_edges, same_dom))
        print(f"week {w.date()}: edges {len(a):,} (full-run {len(b):,}) identical={same_edges}; domain-week features identical={same_dom}", file=LOG, flush=True)
    print("weeks checked:", [str(pd.Timestamp(w).date()) for w in pick]); print("weeks with ANY difference:", bad); assert not bad
tick("causality check")

weeks checked: ['2010-03-22', '2010-03-29', '2010-05-10', '2010-08-02', '2010-10-04', '2010-10-18', '2010-11-29', '2011-01-10', '2011-01-24', '2011-03-07', '2011-03-21', '2011-03-28', '2011-04-18', '2011-05-09', '2011-05-16']
weeks with ANY difference: []


In [6]:
out = dict(chosen_K=chosen, use_email=bool(inp["use_email"]), coverage=inp["cov"], drop_report=inp["drop_rep"], probe={str(k): v for k, v in info.items()})
Path("results").mkdir(exist_ok=True); json.dump(out, open("results/graph_v2_b0.json", "w"), indent=1, default=str)
print("saved results/graph_v2_b0.json; peak RSS GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20, 2))

saved results/graph_v2_b0.json; peak RSS GB: 4.06
